<a href="https://colab.research.google.com/github/VildanaRazumova/thesis-demand-forecasting/blob/main/thesis_notebook_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand-Based Dynamic Pricing Models for Travel Markets

---

# The following Project information



**Client:** A leading B2B tour operator in Kazakhstan and Central Asia, cooperating with 5,000+ travel agencies across destinations including Vietnam, Thailand, Egypt, Turkey, UAE and the Maldives

**Problem statement & Importance:**
Tour operators in B2B travel markets must commit to flight and hotel inventory 6-8 months in advance, without knowing the actual demand. Traditional demand estimation methods rely on historical bookings and expert judgment, which becomes insufficient when market conditions shift due to competitor pricing, seasonal search trends, and external events. This uncertainty leads to either unsold seats or missed revenue opportunities. A critical business challenge is not only whether demand can be predicted accurately, but also how early in the booking window a reliable prediction can be made, giving the operator enough time to act.

**Goal:**
This study focuses on the Kazakhstan → Vietnam route as a pilot, as Vietnam is the operator's most popular destination and each route has unique demand patterns, seasonality, and external signals.

The methodology is designed to be scalable and applicable to other destinations in subsequent research. The primary objective is to develop and evaluate machine learning models that predict flight load factor at multiple booking windows (D-90, D-60, D-30, D-7), and to assess whether external market signals improve prediction accuracy beyond internal booking data alone.

---

Load Factor = Seats Sold / Total Seats x 100%

---

**Business Benefits:**
1. Identifies the earliest reliable prediction window
2. Reduces financial risk from unsold charter seats
3. Provides a data-driven foundation for dynamic price decisions
4. Competitive advantage over operators using rule-based pricing only

**Relevant data collected from:**

Historical Internal signals (2024-2026):
-  booking claims: cancellations, lead time, pax, hotel & flight costs
-  flight capacity: seats sold, empty seats, load factor per window

External signals at specific points in the past:
- Google Trends API: search interest per destination
- Google News API: news per country
- Competitor websites: price monitoring
- Public holiday calendars: event & holiday flags

**Approach:**

A time-series approach is used throughout this project. Features are constructed at each booking window using only information available at that point in time, preventing data leakage and ensuring realistic future deployment.

**Core Task: Load Factor Prediction at multiple booking windows**

| Window | Days Before Departure | Business Meaning |
|--------|----------------------|------------------|
| D-90 | 90 days | Early signal: low booking activity |
| D-60 | 60 days | Search trends become informative |
| D-30 | 30 days | Majority of bookings visible |
| D-7  | 7 days  | Near-final occupancy picture |

For each window two experiments are conducted:
- Baseline model: internal features only (Linear Regression)
- Extended model: internal + external signals (Random Forest, XGBoost, LightGBM)

# Project Pipeline

This project will be approached through the following steps:

1. **Importing the Libraries**: loading all required Python packages
   for data analysis, visualisation and machine learning

2. **Importing the Data**: loading claims and flight load history
   datasets from Google Drive

3. **Data Overview**: understanding the structure, shape, date ranges
   and column descriptions of both datasets

4. **Exploratory Data Analysis (EDA)**: analysing demand patterns,
   seasonality, lead time distribution and booking window behaviour
   for Vietnam routes

5. **Data Preprocessing**: filtering for Vietnam, handling data types,
   dropping irrelevant columns, joining claims with flights

6. **Feature Engineering**: constructing internal features (booking
   pace, lead time, cancellation rate) and external signals
   (price ratio, search trends, holidays) at each booking window
   (D-90, D-60, D-30, D-7)

7. **Model Building**: training baseline models (internal features
   only) and extended models (internal + external features) using
   Linear Regression, Random Forest, XGBoost and LightGBM

8. **Model Evaluation**: comparing model performance across booking
   windows using MAPE, MAE, RMSE and R2

9. **Conclusion & Pipeline Summary**: identifying the most reliable
   prediction window and discussing business implications

# 1. Importing the Libraries

In [5]:
# Import Libraries
from google.colab import drive

# Data libraries
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 2. Import the Data

In [10]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Claims

In [22]:
df_claims = pd.read_csv(
    '/content/drive/MyDrive/Thesis_RazumovaV_2026/thesis_data/_dp_claims_features.csv'
)

In [11]:
df_claims.head(3)

,snapshot_date,snapshot_ts_utc,ClaimID,TourID,StatusID,StatusName,ClaimCreatedDT_UTC,raw_event_time_utc,DepartureDT,TourEndDT,...,FlightPartnerID_Fwd,FlightPartner_Fwd,FlightPartnerID_Bck,FlightPartner_Bck,FlightCost_Fwd,FlightCost_Bck,RevisionAmount_Fwd,RevisionAmount_Bck,CurrencyClaimID,CurrencyAlias
0,2026-03-15,2026-03-15 11:24:39.227,11286829,27194,2,Unpaid,2026-03-05 22:10:00.000,2026-03-05 22:10:00.000,2026-05-02 00:00:00.000,2026-05-10 00:00:00.000,...,157575,SCAT (),157575,SCAT (),596.56,736.56,-70.0,0.0,2,USD
1,2026-03-15,2026-03-15 11:24:39.227,11286196,30853,2,Unpaid,2026-03-05 17:44:00.000,2026-03-05 17:44:00.000,2026-06-04 00:00:00.000,2026-06-14 00:00:00.000,...,157575,SCAT (),157575,SCAT (),928.80,1198.80,-90.0,0.0,2,USD
2,2026-03-15,2026-03-15 11:24:39.227,11286197,30853,2,Unpaid,2026-03-05 17:44:00.000,2026-03-05 17:44:00.000,2026-06-04 00:00:00.000,2026-06-14 00:00:00.000,...,157575,SCAT (),157575,SCAT (),619.20,799.20,-90.0,0.0,2,USD


### Load Flights

In [16]:
df_flights = pd.read_csv(
    '/content/drive/MyDrive/Thesis_RazumovaV_2026/thesis_data/_dp_flight_load_history.csv',
    on_bad_lines='skip'
)

In [17]:
df_flights.head(3)

,snapshot_date,snapshot_ts_utc,FreightID,FlightName,BlockDate,AirlinePartnerID,AirlineName,FlightClassID,ClassAlias,DepartureCityID,...,IsOnRequest,Seats_gross,Sold_gross,Empty,Seats_net,Sold_net,BlockRecords,TotalDaysInSale,ResourceCount,last_stamp
0,2026-03-15,2026-03-15 11:31:47.773,10670,VSV5208,2026-03-05,157575,SCAT (),2,Y,838396,...,0,64,64,0,64,64,2,230,1,\u0000\u0000\u0000\u000b0\u0013Y�
1,2026-03-15,2026-03-15 11:31:47.773,13556,VSV5338,2026-03-05,157575,SCAT (),2,Y,378598,...,0,91,91,0,72,72,1,254,1,\u0000\u0000\u0000\u000b7�n�
2,2026-03-15,2026-03-15 11:31:47.773,13557,VSV5339,2026-03-05,157575,SCAT (),2,Y,293967,...,0,91,91,0,83,83,1,254,1,\u0000\u0000\u0000\u000b3���


# Claims Data Overview

1. General Overview
2. Data types
3. Check duplicates
4. Booking Status Distribution
5. Basic Statistics

In [67]:
# 1. General Overview
print('GENERAL OVERVIEW: df_claims')
print(f'Rows: {df_claims.shape[0]:,}')
print(f'Columns: {df_claims.shape[1]}')
print(f'Booking date: {df_claims["ClaimCreatedDT_UTC"].min()[:10]} - {df_claims["ClaimCreatedDT_UTC"].max()[:10]}')
print(f'Departure: {df_claims["DepartureDT"].min()[:10]} - {df_claims["DepartureDT"].max()[:10]}')
print(f'Destination: {df_claims["CountryName"].unique().tolist()}')
print(f'Missing: {df_claims.isnull().sum().sum()}')

GENERAL OVERVIEW: df_claims
Rows: 56,362
Columns: 63
Booking date: 2024-01-02 - 2026-03-05
Departure: 2024-01-11 - 2026-10-08
Destination: ['Vietnam']
Missing: 0


In [58]:
# 2. Data types
pd.set_option('display.max_rows', 63)
print(df_claims.dtypes)

snapshot_date               object
snapshot_ts_utc             object
ClaimID                      int64
TourID                       int64
StatusID                     int64
StatusName                  object
ClaimCreatedDT_UTC          object
raw_event_time_utc          object
DepartureDT                 object
TourEndDT                   object
ConfirmedDT_UTC             object
LeadTimeDays                 int64
TourNights                   int64
TourNightsCalc               int64
HotelNights                  int64
StateID                      int64
CountryName                 object
TourTypeID                   int64
TourTypeName                object
Padult                       int64
Pchild                       int64
Pinfant                      int64
PaxSeats                     int64
PaxHotel                     int64
HotelID                      int64
HotelName                   object
StarID                       int64
HotelStars                  object
MealID              

In [59]:
# 3. Check duplicates
print(f'Duplicate ClaimIDs:  {df_claims["ClaimID"].duplicated().sum()}')
print(f'Duplicate rows:      {df_claims.duplicated().sum()}')

Duplicate ClaimIDs:  0
Duplicate rows:      0


In [60]:
# 4. Booking Status Distribution
print(df_claims['StatusName'].value_counts())

StatusName
Paid            40162
Canceled        15070
Unpaid           1100
Paid Penalty       21
Penalty             9
Name: count, dtype: int64


In [65]:
print(f'\nCancellation rate: {round(df_claims[df_claims["StatusName"]=="Canceled"].shape[0] / len(df_claims) * 100, 1)}%')
print(f'\nPaid rate: {round(df_claims[df_claims["StatusName"]=="Paid"].shape[0] / len(df_claims) * 100, 1)}%')
print(f'\nUnpaid rate: {round(df_claims[df_claims["StatusName"]=="Unpaid"].shape[0] / len(df_claims) * 100, 1)}%')


Cancellation rate: 26.7%

Paid rate: 71.3%

Unpaid rate: 2.0%


In [62]:
# 5. Basic Statistics
print(df_claims[['LeadTimeDays', 'TourNights', 'Padult', 'Pchild',
                  'HotelCost', 'FlightCost_Fwd', 'FlightCost_Bck',
                  'RevisionAmount_Fwd']].describe().round(2))

       LeadTimeDays  TourNights    Padult   Pchild  HotelCost  FlightCost_Fwd  \
count      56362.00    56362.00  56362.00  56362.0   56362.00        56362.00   
mean          48.27        9.22      2.20      0.3    1059.03          990.55   
std           40.95        2.10      0.83      0.6    1139.03          873.78   
min          -20.00        2.00      1.00      0.0    -122.57         -750.40   
25%           16.00        8.00      2.00      0.0     353.00          519.24   
50%           37.00        9.00      2.00      0.0     724.86          795.22   
75%           71.00       11.00      2.00      0.0    1317.90         1181.53   
max          273.00      105.00     16.00      6.0   22296.28        13610.20   

       FlightCost_Bck  RevisionAmount_Fwd  
count        56362.00            56362.00  
mean           984.07               -9.43  
std            452.82              234.36  
min              0.00            -1250.00  
25%            733.72             -130.00  
50%   

In [69]:
print(df_claims['CurrencyAlias'].value_counts())

CurrencyAlias
USD    56362
Name: count, dtype: int64


The claims dataset contains **56,362 rows** and **63 columns**, covering
the period from **January 2024 to March 2026**. Each row represents a
single tour package claim for the destination **Vietnam**, created by
a travel agency on behalf of an end customer.

The dataset includes the following key information:
- **Booking behaviour:** claim creation date, departure date, lead time
  in days, tour duration, number of nights
- **Passengers:** number of adults, children, and infants per booking
- **Product details:** hotel name, star rating, meal plan, room type,
  accommodation type, hotel region
- **Pricing:** hotel cost, flight cost (forward and return),
  total hotel cost, revision amounts
- **Flight information:** departure and arrival cities, flight class,
  airline partner, freight IDs

**Data Quality:**
- Missing values: **None detected**
- Duplicate ClaimIDs: **0**
- Duplicate rows: **0**

**Booking Status Distribution:**
- Paid: 40,162 (71.3%)
- Unpaid: 1,100 (1.9%)
- Cancellation rate: 1100 (26.7%)

**Key Statistics:**
- Average lead time: 48 days (min: -20, max: 273)

- Average tour duration: 9 nights (min: 2, max: 105)

- Average adults per booking: 2

- Average hotel cost: 1,059 USD (min: -122, max: 22,296)

- Average forward flight cost: 990 USD (min: -750, max: 13,610)

- Average revision amount: -9.43 USD (min: -1,250, max: 4,200)

**⚠️ Anomalies detected:**
- Negative LeadTimeDays (min: -20): bookings created after departure
- Negative HotelCost and FlightCost: likely corrections or refunds
- Extreme outliers in costs: to be investigated in EDA

# Flights Data Overview

1. General Overview
2. Data Types
3. Duplicate Check
4. Basic Statistics
5. Airlines & Routes Distribution
6. Load Factor

In [32]:
# 1. General Overview
print('GENERAL OVERVIEW: df_flights')
print(f'Rows: {df_flights.shape[0]:,}')
print(f'Columns: {df_flights.shape[1]}')
print(f'BlockDate: {df_flights["BlockDate"].min()[:10]} - {df_flights["BlockDate"].max()[:10]}')
print(f'Missing: {df_flights.isnull().sum().sum()}')

GENERAL OVERVIEW: df_flight
Rows: 3,951
Columns: 31
BlockDate: 2024-01-01 - 2026-03-05
Missing: 0


In [49]:
# 2. Data types
pd.set_option('display.max_rows', 31)
print(df_flights.dtypes)

snapshot_date          object
snapshot_ts_utc        object
FreightID               int64
FlightName             object
BlockDate              object
AirlinePartnerID        int64
AirlineName            object
FlightClassID           int64
ClassAlias             object
DepartureCityID         int64
CityFrom               object
ArrivalCityID           int64
CityTo                 object
Country                object
PartnerID               int64
BusinessEntity         object
hard_block              int64
releasedays            object
first_reserve_utc      object
HasStopSale             int64
StopSaleIssueDT_UTC    object
IsOnRequest             int64
Seats_gross             int64
Sold_gross              int64
Empty                   int64
Seats_net               int64
Sold_net                int64
BlockRecords            int64
TotalDaysInSale         int64
ResourceCount           int64
last_stamp             object
dtype: object


In [50]:
# 3. Duplicate Check
print(f'Duplicate FreightIDs: {df_flights["FreightID"].duplicated().sum()}')
print(f'Duplicate rows:       {df_flights.duplicated().sum()}')

Duplicate FreightIDs: 3866
Duplicate rows:       0


In [68]:
# 4. Basic statistics
print(df_flights[['Seats_gross', 'Sold_gross', 'Empty',
                   'TotalDaysInSale']].describe().round(2))

       Seats_gross  Sold_gross    Empty  TotalDaysInSale
count      3951.00     3951.00  3951.00          3951.00
mean         63.37       50.38    12.98           497.96
std          42.58       44.09    31.48           213.78
min           1.00        0.00     0.00            30.00
25%          30.00       20.00     0.00           348.00
50%          45.00       40.00     0.00           428.00
75%          91.00       64.00     3.00           633.00
max         364.00      360.00   183.00           961.00


In [74]:
#5. Airlines & Routes Distribution
print(df_flights['AirlineName'].value_counts())

AirlineName
SCAT ()                               1768
Air Astana ()                         1139
VietJet Air                            714
Qanot Sharq                            299
Selfie Travel (Селфи Тревел) ТОО        23
Pegas Kazakhstan (ПЕГАС КАЗАХСТАН)       3
Touroperator KOMPAS                      2
ANEX Tourism Worldwide DMCC              1
Selfie tour                              1
Crystal Bay Kazakhstan                   1
Name: count, dtype: int64


In [72]:
df_flights['Route'] = df_flights['CityFrom'] + ' -> ' + df_flights['CityTo']
print(df_flights['Route'].value_counts())

Route
Almaty -> Cam Ranh      569
Cam Ranh -> Almaty      557
Phu Quoc -> Almaty      443
Almaty -> Phu Quoc      442
Cam Ranh -> Astana      331
Astana -> Cam Ranh      331
Phu Quoc -> Astana      273
Astana -> Phu Quoc      264
Da Nang -> Astana        79
Astana -> Da Nang        79
Phu Quoc -> Tashkent     77
Tashkent -> Phu Quoc     76
Tashkent -> Cam Ranh     75
Cam Ranh -> Tashkent     74
Almaty -> Da Nang        73
Da Nang -> Almaty        72
Phu Quoc -> Shymkent     29
Shymkent -> Phu Quoc     28
Ural`sk -> Phu Quoc      13
Phu Quoc -> Aktobe       13
Phu Quoc -> Kostanay     13
Phu Quoc -> Ural`sk      13
Aktobe -> Phu Quoc       13
Kostanay -> Phu Quoc     13
Phuket -> Phu Quoc        1
Name: count, dtype: int64


In [73]:
print(df_flights['Country'].value_counts())

Country
Vietnam       1977
Kazakhstan    1823
Uzbekistan     151
Name: count, dtype: int64


In [86]:
# 6. Load Factor
df_flights['LoadFactor'] = (df_flights['Sold_gross'] / df_flights['Seats_gross'] * 100).round(2)
print(df_flights['LoadFactor'].describe().round(2))

count    3951.00
mean       78.18
std        39.30
min         0.00
25%        93.33
50%       100.00
75%       100.00
max       100.00
Name: LoadFactor, dtype: float64


In [88]:
print(f'LF > 100% (sanity check): {(df_flights["LoadFactor"] > 100).sum()}')
print(f'LF = 100% (fully sold):  {(df_flights["LoadFactor"] == 100).sum()}')
print(f'LF = 0%   (empty):       {(df_flights["LoadFactor"] == 0).sum()}')
print(f'LF < 50%  (low demand):  {(df_flights["LoadFactor"] < 50).sum()}')
print(f'LF 50-80% (normal):      {((df_flights["LoadFactor"] >= 50) & (df_flights["LoadFactor"] < 80)).sum()}')
print(f'LF > 80%  (high demand): {(df_flights["LoadFactor"] >= 80).sum()}')

LF > 100% (sanity check): 0
LF = 100% (fully sold):  2622
LF = 0%   (empty):       381
LF < 50%  (low demand):  888
LF 50-80% (normal):      18
LF > 80%  (high demand): 3045


In [89]:
print(df_flights.groupby('AirlineName')['LoadFactor'].mean().round(2).sort_values(ascending=False))

AirlineName
Touroperator KOMPAS                   100.00
Air Astana ()                          99.05
SCAT ()                                78.15
Qanot Sharq                            73.46
VietJet Air                            49.23
Selfie Travel (Селфи Тревел) ТОО       26.09
ANEX Tourism Worldwide DMCC             0.00
Pegas Kazakhstan (ПЕГАС КАЗАХСТАН)      0.00
Crystal Bay Kazakhstan                  0.00
Selfie tour                             0.00
Name: LoadFactor, dtype: float64


The flight load history dataset contains **3,951 rows** and **31 columns**,
covering charter flight blocks operated on Kazakhstan -> Vietnam routes, from **January 2024 to March 2026**.

Each row represents a single charter flight block assigned to the
operator, containing capacity and sales information at the time of
the data snapshot.

The dataset includes the following key information:
- **Flight details:** flight name, block date, airline partner,
  departure and arrival cities, flight class
- **Capacity:** gross seats available, net seats available
- **Demand signals:** seats sold (gross and net), empty seats,
  total days in sale
- **Operational flags:** hard block indicator, stop sale flag,
  on-request flag, release days

**Data Quality:**
- Missing values: **None detected**
- Duplicate rows: **0**
- Duplicate FreightIDs: **3,866** expected, as the same flight
  appears multiple times across different snapshot dates

**Routes Distribution:**
- Main routes: Almaty - Cam Ranh, Almaty - Phu Quoc,
  Astana - Cam Ranh, Astana - Phu Quoc
- Countries covered: Vietnam (1,977), Kazakhstan (1,823),
  Uzbekistan (151)

**TOP 3 Airlines:**
- SCAT: 1,768 flights (44.8%)
- Air Astana: 1,139 flights (28.8%)
- VietJet Air: 714 flights (18.1%)

**Flight Capacity:**
- Average seats per flight: **63**
- Average seats sold: **50**
- Average empty seats: **13**

**Load Factor: Target Variable:**
- Mean Load Factor: **78.18%**
- Median Load Factor: **100%** (majority of flights fully sold)
- Empty flights (LF = 0%): **381 (9.6%)**
- High demand (LF > 80%): **3,045 (77%)**
- Normal demand (LF 50–80%): **18 (0.5%)**
- Low demand (LF < 50%): **888 (22.5%)**

**⚠️ Anomalies detected:**
- High share of LF = 100% - many flights are fully sold
  which may reflect a late snapshot rather than actual demand
- LF = 0% flights (381): likely flights not yet open for sale
- Some airlines show 0% LF: likely new or inactive blocks
- `last_stamp` to be dropped column contains corrupted binary data

# **Exploratory Data Analysis (EDA)**

### EDA for df_claims

1. Lead Time Distribution
2. Booking Pace Over Time
3. Seasonality Patterns
4. Cancellation Pattern
5. Price Distribution

### EDA for df_flights

1. Load Factor Distribution
2. Load Factor by Route
3. Load Factor by Airline
4. Load Factor Over Time